In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from scipy.spatial import cKDTree

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'philadelphia'
STATE_FIPS = '42'
COUNTY_FIPS = '101'
MODEL_TYPE = 'split_rate_4to1_lycd'
LAND_IMPROVEMENT_RATIO = 4.0
MILLAGE = 13.998  # combined city (0.6317%) + school district (0.7681%) = 1.3998%

# GMA zone assignment: static reference extracted from OPA's 2025 GMA PDF
# (parcel centroid → L1/L2/L3 zone labels; 17 / 84 / 613 zones)
GMA_PATH = Path('data/parcel_gma_assignment.parquet')
DOR_SHP_PATH = Path(
    r'C:\projects\LVTShift\cities\philadelphia\data'
    r'\dor_parcels\Philadelphia_DOR_Parcels_202402.shp'
)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)


## Step 1: Load parcel data

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
gdf = gpd.read_parquet(PARCEL_PATH)
gdf['parcel_number'] = gdf['parcel_number'].astype(str).str.zfill(9)
print(f'Loaded {len(gdf):,} parcels')

Loaded 579,815 parcels


## Step 2: Derive lot area from DOR parcel polygons (spatial join)

OPA's `total_area` field is zero for ~32K parcels. Instead we compute area from the
Philadelphia DOR parcel shapefile (PASDA 2024) via point-in-polygon spatial join â€”
matches 99.6% of parcels. The 0.4% remainder get KNN-imputed area in Step 4.

In [3]:
AREA_PATH = DATA_DIR / 'parcel_areas_dor.parquet'

if AREA_PATH.exists():
    dor_areas = pd.read_parquet(AREA_PATH)
    print(f'Loaded DOR area cache: {len(dor_areas):,} rows')
else:
    print('Building DOR area cache via spatial join (one-time, ~30s)...')
    dor = gpd.read_file(DOR_SHP_PATH, columns=['Shape__Are'])
    # DOR shapefile is in EPSG:3857; Shape__Are is in m^2
    pts = gpd.read_parquet(PARCEL_PATH, columns=['parcel_number', 'geometry'])
    pts = pts.to_crs('EPSG:3857')
    joined = gpd.sjoin(pts, dor[['Shape__Are', 'geometry']], how='left', predicate='within')
    joined['dor_area_sqft'] = (joined['Shape__Are'] * 10.7639).round(1)
    dor_areas = joined[['parcel_number', 'dor_area_sqft']].copy()
    dor_areas.to_parquet(AREA_PATH, index=False)
    print(f'  Cached {len(dor_areas):,} rows')

matched = dor_areas['dor_area_sqft'].notna()
print(f'Matched to DOR polygon: {matched.sum():,} ({matched.mean():.1%})')
print(f'Unmatched (needs KNN area imputation): {(~matched).sum():,}')
print(f'area_sqft: median={dor_areas["dor_area_sqft"].median():.0f}, mean={dor_areas["dor_area_sqft"].mean():.0f}')

Loaded DOR area cache: 579,815 rows
Matched to DOR polygon: 577,698 (99.6%)
Unmatched (needs KNN area imputation): 2,117
area_sqft: median=2427, mean=25238


## Step 3: Load GMA zone assignments and join lot area to parcels

In [4]:
gma = pd.read_parquet(GMA_PATH)
gma['key'] = gma['key'].astype(str).str.zfill(9)
print(f'GMA assignment: {len(gma):,} parcels | '
      f'L1={gma["gma1"].nunique()} zones | L2={gma["gma2"].nunique()} | L3={gma["gma3"].nunique()}')

dor_areas['parcel_number'] = dor_areas['parcel_number'].astype(str).str.zfill(9)

# Join DOR lot area
gdf = gdf.merge(dor_areas[['parcel_number', 'dor_area_sqft']], on='parcel_number', how='left')

# Join GMA zone labels
gdf = gdf.merge(
    gma[['key', 'gma1', 'gma2', 'gma3']],
    left_on='parcel_number', right_on='key',
    how='left',
)

gma_matched = gdf['gma3'].notna()
area_matched = gdf['dor_area_sqft'].notna()
print(f'\nParcels with GMA assignment: {gma_matched.sum():,} ({gma_matched.mean():.1%})')
print(f'Parcels with DOR area:        {area_matched.sum():,} ({area_matched.mean():.1%})')
print(f'Parcels needing GMA KNN fallback: {(~gma_matched).sum():,}')
print(f'Parcels needing area imputation:  {(~area_matched).sum():,}')


GMA assignment: 528,204 parcels | L1=17 zones | L2=84 | L3=613



Parcels with GMA assignment: 525,571 (90.6%)
Parcels with DOR area:        577,736 (99.6%)
Parcels needing GMA KNN fallback: 54,282
Parcels needing area imputation:  2,117


## Step 4: Compute GMA hierarchical LYCD land values

For each parcel, land value is estimated using the "Least You Can Do" (LYCD) method
applied at OPA's Geographic Market Area (GMA) zone level:

1. For each GMA zone at the finest applicable level (L3 if ≥ 50 improved parcels,
   else L2, else L1): compute `median(market_value / dor_area_sqft)` over **improved**
   parcels in that zone. "Improved" = OPA category code not in {6, 12, 13}.
2. Apply land allocation: 20% for improved parcels (OPA's standard), 100% for vacant.
3. `lycd_land_value = zone_land_psf × parcel_dor_area_sqft`.

Using `market_value` from the same `assessments WHERE year=2024` pull as the billing
values ensures vintage consistency. The GMA hierarchy (L1: 17 zones, L2: 84 zones,
L3: 613 micro-zones) is OPA's own internal geography for setting assessments.

KNN fallback (5 nearest neighbors) is applied for parcels without a GMA assignment
or without a DOR lot area.

In [5]:
LAND_PCT_IMPROVED = 0.20   # OPA's standard land allocation for improved parcels
LAND_PCT_VACANT   = 1.00   # vacant parcels: market value is all land
MIN_IMPROVED      = 50     # minimum improved parcels per GMA zone before falling back

# Improved vs vacant flag using OPA category codes (same codes as the four-override system)
VACANT_CODES = {'6', '12', '13'}
cat_raw = gdf['category_code'].astype(str).str.strip()
has_market   = gdf['market_value'].notna() & (gdf['market_value'] > 0)
has_area     = gdf['dor_area_sqft'].notna() & (gdf['dor_area_sqft'] > 0)
is_improved  = ~cat_raw.isin(VACANT_CODES) & has_market & has_area
is_vacant_p  =  cat_raw.isin(VACANT_CODES) & has_market & has_area

# Per-parcel total-value PSF (used only to build zone medians for improved parcels)
gdf['_total_psf'] = np.where(
    is_improved | is_vacant_p,
    gdf['market_value'] / gdf['dor_area_sqft'],
    np.nan,
)

# Zone medians of total PSF over improved parcels at each GMA level
imp = gdf[is_improved]
l3_cnt = imp.groupby('gma3')['_total_psf'].count().rename('_l3_n')
l2_cnt = imp.groupby('gma2')['_total_psf'].count().rename('_l2_n')
l3_med = imp.groupby('gma3')['_total_psf'].median().rename('_l3_med')
l2_med = imp.groupby('gma2')['_total_psf'].median().rename('_l2_med')
l1_med = imp.groupby('gma1')['_total_psf'].median().rename('_l1_med')

for col_key, series in [('gma3', l3_cnt), ('gma3', l3_med),
                         ('gma2', l2_cnt), ('gma2', l2_med),
                         ('gma1', l1_med)]:
    gdf = gdf.merge(series.reset_index(), on=col_key, how='left')

# Hierarchical zone-median selection
gdf['_gma_med'] = np.where(
    gdf['_l3_n'].fillna(0) >= MIN_IMPROVED, gdf['_l3_med'],
    np.where(gdf['_l2_n'].fillna(0) >= MIN_IMPROVED, gdf['_l2_med'], gdf['_l1_med'])
)

# Apply land allocation per parcel type
_land_pct = np.where(is_vacant_p, LAND_PCT_VACANT, LAND_PCT_IMPROVED)
_land_psf  = gdf['_gma_med'] * _land_pct

# Land value for GMA-assigned parcels with a DOR area
gdf['lycd_land_value'] = np.where(
    gdf['gma3'].notna() & has_area,
    (_land_psf * gdf['dor_area_sqft']).clip(lower=0),
    np.nan,
)

# Track GMA level used
gdf['gma_level'] = np.where(
    gdf['gma3'].isna(), 'knn',
    np.where(gdf['_l3_n'].fillna(0) >= MIN_IMPROVED, 'L3',
    np.where(gdf['_l2_n'].fillna(0) >= MIN_IMPROVED, 'L2', 'L1'))
)

n_gma       = int(gdf['lycd_land_value'].notna().sum())
n_knn_needed = int(gdf['lycd_land_value'].isna().sum())
print(f'GMA land values computed:      {n_gma:,}')
print(f'Parcels needing KNN fallback:  {n_knn_needed:,}')
print()

# Project to EPSG:3857 for KNN
gdf_m  = gdf.to_crs('EPSG:3857')
coords = np.column_stack([gdf_m.geometry.x, gdf_m.geometry.y])

def knn_impute(values, coords, K=5):
    values = np.asarray(values, dtype=float)
    has_v  = ~np.isnan(values)
    need_v = np.isnan(values)
    if need_v.sum() == 0:
        return values
    tree = cKDTree(coords[has_v])
    _, idxs = tree.query(coords[need_v], k=min(K, int(has_v.sum())), workers=-1)
    out = values.copy()
    out[need_v] = np.median(values[has_v][idxs], axis=1)
    return out

# KNN-impute DOR area for parcels without a polygon hit
n_area_missing = int(gdf['dor_area_sqft'].isna().sum())
gdf['dor_area_sqft'] = knn_impute(gdf['dor_area_sqft'].values.astype(float), coords)
print(f'KNN-imputed dor_area_sqft for {n_area_missing:,} parcels')

# KNN-impute lycd_land_value for parcels outside GMA coverage
gdf['lycd_land_value'] = knn_impute(gdf['lycd_land_value'].values, coords)
print(f'KNN-imputed lycd_land_value for {n_knn_needed:,} parcels (no GMA zone or DOR area)')

gdf.loc[gdf['gma_level'] == 'knn', 'gma_level'] = 'knn'

# Clean up temporaries
gdf.drop(columns=['_total_psf', '_gma_med', '_l3_n', '_l2_n',
                   '_l3_med', '_l2_med', '_l1_med'], inplace=True)

print()
print('GMA level used:')
print(gdf['gma_level'].value_counts().to_string())
print()
print(f'All parcels have lycd_land_value: {gdf["lycd_land_value"].isna().sum() == 0}')


GMA land values computed:      525,449
Parcels needing KNN fallback:  54,404



KNN-imputed dor_area_sqft for 2,117 parcels


KNN-imputed lycd_land_value for 54,404 parcels (no GMA zone or DOR area)

GMA level used:
gma_level
L3     524563
knn     54282
L2        906
L1        102

All parcels have lycd_land_value: True


## Step 5: Summarize LYCD land base

In [6]:
total_lycd_land = gdf['lycd_land_value'].sum()
total_opa_land  = gdf['taxable_land'].sum()
print(f'Total LYCD land base:    ${total_lycd_land/1e9:.2f}B')
print(f'Total OPA taxable land:  ${total_opa_land/1e9:.2f}B')
print(f'Ratio LYCD/OPA:          {total_lycd_land/total_opa_land:.2f}x')
print()
print('LYCD land value by GMA level (median $):')
print(gdf.groupby('gma_level')['lycd_land_value'].median().sort_values(ascending=False).to_string())
print()
print('LYCD land value percentiles (all parcels):')
for p in [10, 25, 50, 75, 90, 99]:
    v = gdf['lycd_land_value'].quantile(p/100)
    print(f'  p{p:2d}: ${v:,.0f}')


Total LYCD land base:    $91.56B
Total OPA taxable land:  $36.63B
Ratio LYCD/OPA:          2.50x

LYCD land value by GMA level (median $):
gma_level
L2     1.262402e+06
L1     1.164806e+06
knn    1.364513e+05
L3     3.776519e+04

LYCD land value percentiles (all parcels):
  p10: $15,345
  p25: $24,055
  p50: $40,427
  p75: $71,530
  p90: $155,278
  p99: $2,139,320


## Step 6: Categorize parcels (same overrides as OPA model)

In [7]:
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement -> Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: abated improved parcels
GENUINE_VACANT_CODES = {'6', '12', '13'}
abated_mask = (
    (gdf['taxable_building'] <= 0) &
    (gdf['taxable_land'] > 0) &
    (~gdf['category_code'].isin(GENUINE_VACANT_CODES))
)
gdf.loc[abated_mask, 'PROPERTY_CATEGORY'] = 'Abated / Construction Exemption'

# Override 3: OPA-vacant with nonzero building value
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) &
    (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: fully exempt parcels
EXEMPT_CATEGORY_MAP = {k: v + ' â€” Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code']
    .map(EXEMPT_CATEGORY_MAP)
    .fillna('Other â€” Exempt')
)

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable: {(gdf["full_exmp"] == 0).sum():,}')
print()
print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

Total parcels: 579,853
Fully exempt: 44,134  |  Abated: 30,129  |  Improved vacant: 1,485  |  Taxable: 535,719

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential                    408141
Small Multi-Family (2-4 units)                37944
Abated / Construction Exemption               30129
Vacant Land                                   27927
Single Family Residential â€” Exempt          27059
Mixed Use                                     13399
Vacant Land â€” Exempt                        11860
Commercial                                     8521
Industrial                                     3489
Commercial â€” Exempt                          3280
Large Multi-Family (5+ units)                  2547
Improved Vacant Land                           1485
Small Multi-Family (2-4 units) â€” Exempt      1114
Other Residential                              1068
Office / Commercial Condo                       805
Large Multi-Family (5+ units) â€” Exempt        300
Indust

## Step 7: Current tax (OPA taxable values â€” revenue baseline)

In [8]:
gdf['millage_rate'] = MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

city_only_rate = 6.317  # mills
city_revenue = gdf['taxable_total'].mul(city_only_rate / 1000).sum()
gap_pct = (city_revenue / 795_796_126 - 1) * 100

print(f'Modeled combined levy (city + school):  ${current_revenue:,.0f}')
print(f'Implied city-only portion (0.6317%):    ${city_revenue:,.0f}')
print(f'FY2024 city-only actuals (approx):      $795,796,126')
print(f'City portion gap: {gap_pct:+.1f}%  (target: within Â±5%)')
assert abs(gap_pct) < 10.0, f'City gap {gap_pct:.1f}% exceeds 10%'

Modeled combined levy (city + school):  $1,851,235,135
Implied city-only portion (0.6317%):    $835,423,086
FY2024 city-only actuals (approx):      $795,796,126
City portion gap: +5.0%  (target: within Â±5%)


## Step 8: Build GMA LYCD reform base

Land value: GMA hierarchical LYCD (`lycd_land_value` — zone-median OPA market value × 20%
× lot area for GMA-assigned parcels; KNN-imputed for the remainder).
Building value: OPA `taxable_building` — preserves existing Homestead and abatement exemption structure.

For abated parcels (OPA shows zero building): impute building = 4 × LYCD land value
(restoring OPA's implicit 20% land ratio applied to the LYCD-estimated land base).

In [9]:
IMPUTED_BLDG_MULTIPLIER = 4.0

gdf['model_land'] = gdf['lycd_land_value'].clip(lower=0)
gdf['model_building'] = gdf['taxable_building'].clip(lower=0)

abated = gdf['PROPERTY_CATEGORY'] == 'Abated / Construction Exemption'
gdf.loc[abated, 'model_building'] = gdf.loc[abated, 'model_land'] * IMPUTED_BLDG_MULTIPLIER

n_imputed = abated.sum()
imputed_bldg_total = gdf.loc[abated, 'model_building'].sum()
print(f'Abated parcels with imputed building value: {n_imputed:,}')
print(f'Total imputed building base added to reform:  ${imputed_bldg_total/1e9:.2f}B')
print()
print(f'Reform land base:          ${gdf["model_land"].sum()/1e9:.2f}B')
print(f'Reform improvement base:   ${gdf["model_building"].sum()/1e9:.2f}B')
print(f'OPA taxable land base:     ${gdf["taxable_land"].sum()/1e9:.2f}B')
print(f'OPA taxable building base: ${gdf["taxable_building"].sum()/1e9:.2f}B')

Abated parcels with imputed building value: 30,129
Total imputed building base added to reform:  $15.16B

Reform land base:          $91.56B
Reform improvement base:   $110.78B
OPA taxable land base:     $36.63B
OPA taxable building base: $95.62B


## Step 9: Revenue-neutral split-rate model (4:1 land:improvement)

In [10]:
taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='model_land',
    improvement_value_col='model_building',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine exempt parcels
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
exempt['taxable_land_value'] = 0.0
exempt['taxable_improvement_value'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} mills')
print(f'Improvement millage: {improvement_millage:.4f} mills')
print(f'Revenue check:       ${new_revenue:,.0f} (target: ${taxable["current_tax"].sum():,.0f})')
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Philadelphia â€” 4:1 Split-Rate Tax Impact (LYCD Land Values)')

Land millage:        18.6971 mills
Improvement millage: 4.6743 mills
Revenue check:       $1,851,235,135 (target: $1,851,235,135)




Philadelphia â€” 4:1 Split-Rate Tax Impact (LYCD Land Values)
                                 Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
                Single Family Residential 408141    $-24,475,595       -2.4%       $-60        $-666    1.7%     -39.1%             9.7%            83.8%
           Small Multi-Family (2-4 units)  37944   $-103,784,116      -47.4%    $-2,735      $-1,525  -40.6%     -47.1%             3.2%            93.9%
          Abated / Construction Exemption  30129     $97,927,885      223.6%     $3,250         $601 1441.0%     245.8%            85.4%             8.9%
                              Vacant Land  27927    $182,823,511      536.8%     $6,546       $1,392 1427.3%     381.2%            96.7%             2.7%
     Single Family Residential â€” Exempt  27059              $0        0.0%         $0           $0    0.0%       0.0%             0.0%             0.0%
             

## Step 10: Census join

In [11]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f'Census join: {gdf["std_geoid"].notna().mean()*100:.1f}% matched')
        except concurrent.futures.TimeoutError:
            print('Census API timed out â€” skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


## Step 11: Export and visualize

In [12]:
out_df = save_standard_export(
    df=gdf,
    city=f'{CITY_NAME}_lycd',
    output_path=f'../../analysis/data/{CITY_NAME}_lycd.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

from lvt.viz import create_city_report
create_city_report(out_df, f'{CITY_NAME}_lycd', show=False)
print('Done.')

  [warn] philadelphia_lycd: non-standard property categories (will be preserved): ['Abated / Construction Exemption', 'Commercial â€” Exempt', 'Hotel â€” Exempt', 'Improved Vacant Land', 'Industrial â€” Exempt', 'Large Multi-Family (5+ units) â€” Exempt', 'Mixed Use â€” Exempt', 'Office / Commercial Condo â€” Exempt', 'Other Commercial â€” Exempt', 'Other Residential â€” Exempt', 'Other â€” Exempt', 'Single Family Residential â€” Exempt', 'Small Multi-Family (2-4 units) â€” Exempt', 'Vacant Land â€” Exempt']


  ✓ philadelphia_lycd: 579,853 rows → ../../analysis/data/philadelphia_lycd.csv  [model: split_rate_4to1_lycd]


Done.
